In [ ]:
!pip install -U transformers accelerate safetensors huggingface_hub

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-32B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

In [2]:
from pathlib import Path

PROJECT_ROOT = Path("storyfx")

excerpt = "excerpt_000"
task = "segment_specification"
strategy = "storyfx"

prompt_path = PROJECT_ROOT / "configs" / "prompts" / strategy / f"{task}.txt"
excerpt_path = PROJECT_ROOT / "data" / excerpt /  "raw" / f"{excerpt}.txt"

prompt_template = prompt_path.read_text(encoding="utf-8").strip()
excerpt_text = excerpt_path.read_text(encoding="utf-8").strip()

prompt = (
    f"{prompt_template}\n\n"
    f"### INPUT NARRATIVE\n\n"
    f"{excerpt_text}"
)

messages = [
    {"role": "user", "content": prompt}
]

inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=1200,
    do_sample=False
)

response = tokenizer.decode(
    outputs[0][inputs["input_ids"].shape[-1]:],
    skip_special_tokens=True
).strip()

print(response)

parsed_response = json.loads(response)


output_path.write_text(
    json.dumps(parsed_response, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print(f"Specification saved to: {output_path}")

FileNotFoundError: [Errno 2] No such file or directory: 'storyfx\\configs\\prompts\\storyfx\\segment_specification.txt'